<a href="https://colab.research.google.com/github/mbiriri/Autonomous_Trailer/blob/main/Autonomous_Trailer_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Autonomous Trailer Knowledge-Based Agent


## Step 1: Identify the Task
Build a knowledge-based agent for an autonomous freight trailer moving along a directed road network (Mombasa to Eldoret / Kisumu). It uses A* search tree planning, distance metrics, live weather/hub data, and safety rules to pick routes and handle road or signal conditions safely.

## Step 2: Assemble Relevant Knowledge
What the agent needs:
- Traffic signal and road safety rules (signals, weighbridges, steep grades, bridge height/weight limits).
- A directed weighted graph of hubs and road segments.
- Distance measures: **Euclidean**, **Manhattan**, and **Haversine**.
- External data via **Open-Meteo** (weather) and **OpenStreetMap Nominatim** (hub lookup), with a local fallback if offline.

In [1]:
import math
import heapq
import json
import urllib.request
import urllib.parse
from datetime import datetime
from typing import Dict, List, Tuple, Optional, Any

try:
    from bs4 import BeautifulSoup
    BS4_AVAILABLE = True
except ImportError:
    BS4_AVAILABLE = False

print(f"Environment initialized at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"BeautifulSoup4 available for web scraping: {BS4_AVAILABLE}")

Environment initialized at 2026-11-19 17:56:50
BeautifulSoup4 available for web scraping: True


### Fetching Weather & Hub Data

In [2]:
class ExternalKnowledgeHarvester:
    """
    Fetches real-time environmental and geospatial facts via public APIs
    and permitted web scraping to enrich the trailer's Knowledge Base.
    """
    def __init__(self, timeout: int = 5):
        self.timeout = timeout
        self.headers = {"User-Agent": "AutonomousTrailerKBA-StudentProject/1.0"}

    def fetch_live_weather(self, lat: float, lon: float, location_name: str) -> Dict[str, Any]:
        """Queries Open-Meteo API for real-time weather conditions at a logistics hub."""
        url = (
            f"https://api.open-meteo.com/v1/forecast?"
            f"latitude={lat}&longitude={lon}&current=temperature_2m,precipitation,wind_speed_10m,weather_code"
        )
        try:
            req = urllib.request.Request(url, headers=self.headers)
            with urllib.request.urlopen(req, timeout=self.timeout) as resp:
                data = json.loads(resp.read().decode("utf-8"))
                current = data.get("current", {})
                precip = current.get("precipitation", 0.0)
                wind = current.get("wind_speed_10m", 0.0)
                temp = current.get("temperature_2m", 22.0)

                condition = "Clear"
                if precip > 2.0:
                    condition = "Heavy_Rain"
                elif precip > 0.0:
                    condition = "Wet_Road"
                elif wind > 45.0:
                    condition = "High_Crosswind"

                return {
                    "location": location_name,
                    "source": "Open-Meteo Live API",
                    "temperature_c": temp,
                    "precipitation_mm": precip,
                    "wind_kmh": wind,
                    "road_weather_state": condition
                }
        except Exception:
            # Graceful deterministic fallback when offline
            return {
                "location": location_name,
                "source": "Cached Telemetry Fallback",
                "temperature_c": 21.5,
                "precipitation_mm": 0.0,
                "wind_kmh": 14.2,
                "road_weather_state": "Clear"
            }

    def fetch_hub_metadata(self, query: str) -> Dict[str, Any]:
        """Queries OpenStreetMap Nominatim API for hub verification and bounding metadata."""
        encoded = urllib.parse.quote(query)
        url = f"https://nominatim.openstreetmap.org/search?q={encoded}&format=json&limit=1"
        try:
            req = urllib.request.Request(url, headers=self.headers)
            with urllib.request.urlopen(req, timeout=self.timeout) as resp:
                results = json.loads(resp.read().decode("utf-8"))
                if results:
                    return {
                        "query": query,
                        "verified_name": results[0].get("display_name", query),
                        "lat": float(results[0]["lat"]),
                        "lon": float(results[0]["lon"]),
                        "source": "OSM Nominatim API"
                    }
        except Exception:
            pass
        return {"query": query, "verified_name": query, "source": "Local Corridor Registry"}

harvester = ExternalKnowledgeHarvester()
sample_weather = harvester.fetch_live_weather(-1.2921, 36.8219, "Nairobi_Inland_Depot")
print("Sample API Knowledge Retrieved:", json.dumps(sample_weather, indent=2))

Sample API Knowledge Retrieved: {
  "location": "Nairobi_Inland_Depot",
  "source": "Open-Meteo Live API",
  "temperature_c": 17.6,
  "precipitation_mm": 0.6,
  "wind_kmh": 13.8,
  "road_weather_state": "Wet_Road"
}


## Step 3: Define the Vocabulary
- **Nodes:** `Mombasa_Port`, `Voi_Weighbridge`, `Mtito_Andei_Junction`, `Machakos_Turnoff`, `Nairobi_Inland_Depot`, `Nairobi_Southern_Bypass`, `Naivasha_Dry_Port`, `Nakuru_Hub`, `Eldoret_Terminal`, `Kisumu_Port`
- **Signals:** `Red`, `Yellow`, `Green`, `Flashing_Red`, `Flashing_Amber`, `None`
- **Weather:** `Clear`, `Wet_Road`, `Heavy_Rain`, `Fog`, `High_Crosswind`
- **Predicates / Relations:** `Light(color)`, `Connected(u, v, dist, speed, max_tons, clearance)`, `WeighbridgeAt(node)`, `WeatherAt(node, cond)`, `CanTraverse(trailer, u, v)`, `Action(state, cmd)`

## Step 4: Encode General Knowledge

In [3]:
class DistanceMetrics:
    """Implements distance measures required for spatial reasoning and A* heuristics."""

    @staticmethod
    def euclidean(p1: Tuple[float, float], p2: Tuple[float, float]) -> float:
        """L2 straight-line distance in planar coordinates (km)."""
        return math.sqrt((p1[0] - p2[0]) ** 2 + (p1[1] - p2[1]) ** 2)

    @staticmethod
    def manhattan(p1: Tuple[float, float], p2: Tuple[float, float]) -> float:
        """L1 grid distance for urban depot/yard navigation (km)."""
        return abs(p1[0] - p2[0]) + abs(p1[1] - p2[1])

    @staticmethod
    def haversine(coord1: Tuple[float, float], coord2: Tuple[float, float]) -> float:
        """Great-circle distance between (lat, lon) points in kilometers."""
        R = 6371.0
        lat1, lon1 = map(math.radians, coord1)
        lat2, lon2 = map(math.radians, coord2)
        dlat = lat2 - lat1
        dlon = lon2 - lon1
        a = math.sin(dlat / 2) ** 2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2) ** 2
        c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
        return R * c


class SearchTreeNode:
    """Represents a node in the search tree generated during pathfinding."""
    def __init__(self, state: str, parent: Optional['SearchTreeNode'] = None,
                 action: Optional[str] = None, g_cost: float = 0.0, h_cost: float = 0.0):
        self.state = state
        self.parent = parent
        self.action = action
        self.g_cost = g_cost
        self.h_cost = h_cost
        self.f_cost = g_cost + h_cost
        self.depth = 0 if parent is None else parent.depth + 1

    def __lt__(self, other: 'SearchTreeNode') -> bool:
        return self.f_cost < other.f_cost

    def reconstruct_path(self) -> List[str]:
        path = []
        curr = self
        while curr is not None:
            path.append(curr.state)
            curr = curr.parent
        return list(reversed(path))


class DirectedCorridorGraph:
    """
    Directed weighted graph representing the autonomous trailer's road network.
    Each edge encodes distance, speed limit, weight capacity, bridge clearance, and status.
    """
    def __init__(self):
        self.nodes: Dict[str, Dict[str, Any]] = {}
        self.edges: Dict[str, List[Dict[str, Any]]] = {}

    def add_node(self, name: str, lat: float, lon: float, grid_xy: Tuple[float, float],
                 node_type: str = "Hub", weighbridge: bool = False):
        self.nodes[name] = {
            "lat": lat,
            "lon": lon,
            "grid_xy": grid_xy,
            "type": node_type,
            "weighbridge": weighbridge,
            "weather": "Clear"
        }
        self.edges.setdefault(name, [])

    def add_directed_edge(self, u: str, v: str, distance_km: float, speed_limit: int = 80,
                          max_tons: float = 50.0, clearance_m: float = 5.2,
                          steep_grade: bool = False, open_road: bool = True):
        if u not in self.nodes or v not in self.nodes:
            raise ValueError(f"Both nodes {u} and {v} must exist in the graph.")
        self.edges[u].append({
            "to": v,
            "distance_km": distance_km,
            "speed_limit": speed_limit,
            "max_tons": max_tons,
            "clearance_m": clearance_m,
            "steep_grade": steep_grade,
            "open_road": open_road
        })

    def get_heuristic(self, u: str, goal: str, metric: str = "haversine") -> float:
        n1, n2 = self.nodes[u], self.nodes[goal]
        if metric == "haversine":
            return DistanceMetrics.haversine((n1["lat"], n1["lon"]), (n2["lat"], n2["lon"]))
        elif metric == "euclidean":
            return DistanceMetrics.euclidean(n1["grid_xy"], n2["grid_xy"])
        elif metric == "manhattan":
            return DistanceMetrics.manhattan(n1["grid_xy"], n2["grid_xy"])
        raise ValueError(f"Unknown metric: {metric}")


TRAFFIC_SIGNAL_RULES = {
    "Red": "FULL_STOP — Engage air brakes and hold behind stop line.",
    "Yellow": "DECELERATE_TO_STOP — Apply engine retarder; stop unless within minimum heavy-trailer braking distance.",
    "Green": "PROCEED — Scan cross-traffic, verify trailer swing clearance, and maintain target speed.",
    "Flashing_Red": "STOP_AND_YIELD — Come to a complete stop, verify intersection clear, then proceed.",
    "Flashing_Amber": "PROCEED_WITH_CAUTION — Reduce speed by 30% and monitor pedestrian/merging traffic.",
    "None": "UNCONTROLLED_INTERSECTION — Treat as yield; slow to 25 km/h and scan sensors."
}

WEATHER_SPEED_MULTIPLIERS = {
    "Clear": 1.0,
    "Wet_Road": 0.75,
    "Heavy_Rain": 0.55,
    "Fog": 0.50,
    "High_Crosswind": 0.65
}

print("Domain vocabulary, distance metrics, directed graph, and base rules encoded successfully.")

Domain vocabulary, distance metrics, directed graph, and base rules encoded successfully.


## Step 5: Add Facts & Build the Knowledge Base

In [4]:
class TrailerKnowledgeBase:
    """
    Central Knowledge Base for the Autonomous Self-Driving Trailer.
    Combines symbolic rules, dynamic percept facts, and a spatial directed graph.
    """
    def __init__(self):
        self.signal_rules: Dict[str, str] = dict(TRAFFIC_SIGNAL_RULES)
        self.weather_multipliers: Dict[str, float] = dict(WEATHER_SPEED_MULTIPLIERS)
        self.custom_rules: Dict[str, str] = {}
        self.graph = DirectedCorridorGraph()
        self.trailer_profile: Dict[str, Any] = {
            "id": "TRL-KE-001",
            "gross_weight_tons": 38.0,
            "height_m": 4.3,
            "length_m": 16.5,
            "cargo_type": "Standard_Load"
        }
        self.current_percepts: Dict[str, Any] = {}

    def add_new_fact(self, category: str, key: str, value: Any):
        """TELL operation: Add or update a fact/rule in the Knowledge Base."""
        cat = category.lower()
        if cat == "signal":
            self.signal_rules[key] = value
        elif cat == "weather":
            self.weather_multipliers[key] = float(value)
        elif cat == "rule":
            self.custom_rules[key] = value
        elif cat == "percept":
            self.current_percepts[key] = value
        elif cat == "trailer":
            self.trailer_profile[key] = value
        else:
            self.custom_rules[f"{category}:{key}"] = str(value)
        print(f"[KB TELL] Added/Updated [{category}] '{key}' -> {value}")

    def set_road_status(self, u: str, v: str, open_road: bool, reason: str = ""):
        """Dynamically updates directed edge availability (e.g., accident or construction)."""
        for edge in self.graph.edges.get(u, []):
            if edge["to"] == v:
                edge["open_road"] = open_road
                state = "OPEN" if open_road else f"CLOSED ({reason})"
                print(f"[KB GRAPH UPDATE] Directed edge {u} -> {v} is now {state}")
                return
        print(f"[KB WARNING] Edge {u} -> {v} not found.")

    def show_kb(self):
        """Displays a structured summary of the current Knowledge Base."""
        print("\n==================== AUTONOMOUS TRAILER KNOWLEDGE BASE ====================")
        print(f"Trailer Specs : {self.trailer_profile}")
        print(f"Graph Nodes   : {len(self.graph.nodes)} hubs | Directed Edges: {sum(len(e) for e in self.graph.edges.values())}")
        print("\n--- Traffic Signal Rules ---")
        for sig, act in self.signal_rules.items():
            print(f"  {sig:15} => {act}")
        print("\n--- Custom Domain & Safety Rules ---")
        for k, v in self.custom_rules.items():
            print(f"  {k:22} => {v}")
        print("===========================================================================\n")


kb = TrailerKnowledgeBase()

corridor_nodes = [
    ("Mombasa_Port",            -4.0435, 39.6682, (0.0, 0.0),    "Port_Origin", True),
    ("Voi_Weighbridge",         -3.3961, 38.5561, (140.0, 70.0), "Weighbridge", True),
    ("Mtito_Andei_Junction",    -2.6897, 38.1667, (235.0, 145.0),"Junction",    False),
    ("Machakos_Turnoff",        -1.5833, 37.2667, (390.0, 265.0),"Junction",    False),
    ("Nairobi_Inland_Depot",    -1.3300, 36.8800, (445.0, 295.0),"Inland_Port", True),
    ("Nairobi_Southern_Bypass", -1.3200, 36.7500, (440.0, 305.0),"Bypass",      False),
    ("Naivasha_Dry_Port",       -0.7172, 36.4310, (525.0, 375.0),"Dry_Port",    False),
    ("Nakuru_Hub",              -0.3031, 36.0800, (585.0, 420.0),"Hub",         True),
    ("Eldoret_Terminal",         0.5143, 35.2698, (710.0, 515.0),"Terminal",    False),
    ("Kisumu_Port",             -0.0917, 34.7680, (760.0, 445.0),"Lake_Port",   False),
]

for name, lat, lon, gxy, ntype, wb in corridor_nodes:
    kb.graph.add_node(name, lat, lon, gxy, node_type=ntype, weighbridge=wb)

directed_routes = [
    ("Mombasa_Port", "Voi_Weighbridge", 155.0, 80, 56.0, 5.5, False, True),
    ("Voi_Weighbridge", "Mtito_Andei_Junction", 102.0, 80, 56.0, 5.5, False, True),
    ("Mtito_Andei_Junction", "Machakos_Turnoff", 185.0, 80, 56.0, 5.2, False, True),
    ("Machakos_Turnoff", "Nairobi_Inland_Depot", 52.0, 70, 56.0, 5.0, False, True),
    ("Machakos_Turnoff", "Nairobi_Southern_Bypass", 58.0, 80, 50.0, 4.6, False, True),
    ("Nairobi_Inland_Depot", "Nairobi_Southern_Bypass", 18.0, 60, 50.0, 4.6, False, True),
    ("Nairobi_Southern_Bypass", "Naivasha_Dry_Port", 78.0, 65, 56.0, 5.4, True, True),
    ("Naivasha_Dry_Port", "Nakuru_Hub", 72.0, 80, 56.0, 5.4, False, True),
    ("Nakuru_Hub", "Eldoret_Terminal", 156.0, 80, 56.0, 5.2, True, True),
    ("Nakuru_Hub", "Kisumu_Port", 184.0, 75, 48.0, 5.0, False, True),
    ("Eldoret_Terminal", "Kisumu_Port", 122.0, 70, 48.0, 5.0, False, True),
    ("Nakuru_Hub", "Naivasha_Dry_Port", 72.0, 80, 56.0, 5.4, False, True),
    ("Naivasha_Dry_Port", "Nairobi_Southern_Bypass", 78.0, 60, 56.0, 5.4, True, True),
]

for edge in directed_routes:
    kb.graph.add_directed_edge(*edge)

kb.add_new_fact("rule", "Weighbridge_Mandatory", "All trailers > 3.5 tons MUST enter weighbridge lane at 15 km/h.")
kb.add_new_fact("rule", "Steep_Grade_Brake_Rule", "Engage low gear + Jake brake; cap speed at 45 km/h to prevent jackknifing.")
kb.add_new_fact("rule", "Bridge_Clearance_Margin", "Require minimum 0.25m safety buffer above trailer height.")
kb.add_new_fact("rule", "Hazardous_Cargo_Curfew", "Hazardous cargo requires escort and avoids urban center transit.")

for hub_name in ["Mombasa_Port", "Nairobi_Inland_Depot", "Nakuru_Hub", "Eldoret_Terminal"]:
    node = kb.graph.nodes[hub_name]
    weather_info = harvester.fetch_live_weather(node["lat"], node["lon"], hub_name)
    kb.graph.nodes[hub_name]["weather"] = weather_info["road_weather_state"]
    print(f"[API ENRICHMENT] {hub_name}: {weather_info['road_weather_state']} "
          f"({weather_info['temperature_c']}°C, {weather_info['source']})")

kb.show_kb()

[KB TELL] Added/Updated [rule] 'Weighbridge_Mandatory' -> All trailers > 3.5 tons MUST enter weighbridge lane at 15 km/h.
[KB TELL] Added/Updated [rule] 'Steep_Grade_Brake_Rule' -> Engage low gear + Jake brake; cap speed at 45 km/h to prevent jackknifing.
[KB TELL] Added/Updated [rule] 'Bridge_Clearance_Margin' -> Require minimum 0.25m safety buffer above trailer height.
[KB TELL] Added/Updated [rule] 'Hazardous_Cargo_Curfew' -> Hazardous cargo requires escort and avoids urban center transit.
[API ENRICHMENT] Mombasa_Port: Clear (27.3°C, Open-Meteo Live API)
[API ENRICHMENT] Nairobi_Inland_Depot: Wet_Road (18.4°C, Open-Meteo Live API)
[API ENRICHMENT] Nakuru_Hub: Clear (18.3°C, Open-Meteo Live API)
[API ENRICHMENT] Eldoret_Terminal: Clear (19.6°C, Open-Meteo Live API)

==================== AUTONOMOUS TRAILER KNOWLEDGE BASE ====================
Trailer Specs : {'id': 'TRL-KE-001', 'gross_weight_tons': 38.0, 'height_m': 4.3, 'length_m': 16.5, 'cargo_type': 'Standard_Load'}
Graph Nodes   

## Steps 6 & 7: Inference, Search Tree & Queries

In [5]:
class AutonomousTrailerAgent:
    """
    Knowledge-Based Agent that combines rule-based logical inference with
    A* search-tree planning over a directed weighted graph.
    """
    def __init__(self, knowledge_base: TrailerKnowledgeBase):
        self.kb = knowledge_base

    def can_traverse_edge(self, edge: Dict[str, Any]) -> Tuple[bool, str]:
        """Evaluates physical and regulatory constraints on a directed edge."""
        if not edge["open_road"]:
            return False, "Road closed"

        trailer_weight = self.kb.trailer_profile["gross_weight_tons"]
        if trailer_weight > edge["max_tons"]:
            return False, f"Overweight ({trailer_weight}t > {edge['max_tons']}t limit)"

        trailer_height = self.kb.trailer_profile["height_m"]
        required_clearance = trailer_height + 0.25
        if required_clearance > edge["clearance_m"]:
            return False, f"Insufficient clearance ({required_clearance:.2f}m > {edge['clearance_m']}m bridge)"

        return True, "OK"

    def compute_safe_edge_speed(self, u: str, edge: Dict[str, Any]) -> float:
        """Infers safe operating speed based on weather, steep grades, and cargo."""
        base_speed = float(edge["speed_limit"])
        weather = self.kb.graph.nodes[u].get("weather", "Clear")
        multiplier = self.kb.weather_multipliers.get(weather, 0.7)
        safe_speed = base_speed * multiplier

        if edge["steep_grade"]:
            safe_speed = min(safe_speed, 45.0)
        if self.kb.trailer_profile["cargo_type"] == "Hazardous_Cargo":
            safe_speed = min(safe_speed, 60.0)
        return round(safe_speed, 1)

    def plan_route(self, start: str, goal: str, metric: str = "haversine",
                   optimize_by: str = "distance") -> Dict[str, Any]:
        """
        A* Search Tree implementation over the directed graph.
        Supports 'distance' (km) or 'time' (hours) cost optimization, and
        'haversine', 'euclidean', or 'manhattan' heuristics.
        """
        if start not in self.kb.graph.nodes or goal not in self.kb.graph.nodes:
            return {"success": False, "reason": "Invalid start or goal node."}

        h_start = self.kb.graph.get_heuristic(start, goal, metric)
        if optimize_by == "time":
            h_start = h_start / 80.0

        root = SearchTreeNode(state=start, g_cost=0.0, h_cost=h_start)
        frontier: List[SearchTreeNode] = [root]
        best_g: Dict[str, float] = {start: 0.0}
        expanded_nodes: List[Tuple[str, float, float, float]] = []
        pruned_edges: List[str] = []

        while frontier:
            current = heapq.heappop(frontier)
            expanded_nodes.append((current.state, round(current.g_cost, 2),
                                   round(current.h_cost, 2), round(current.f_cost, 2)))

            if current.state == goal:
                path = current.reconstruct_path()
                return {
                    "success": True,
                    "start": start,
                    "goal": goal,
                    "metric": metric,
                    "optimize_by": optimize_by,
                    "path": path,
                    "total_cost": round(current.g_cost, 2),
                    "nodes_expanded": len(expanded_nodes),
                    "search_tree_trace": expanded_nodes,
                    "pruned_edges": pruned_edges
                }

            for edge in self.kb.graph.edges.get(current.state, []):
                neighbor = edge["to"]
                allowed, reason = self.can_traverse_edge(edge)
                if not allowed:
                    pruned_edges.append(f"{current.state} -> {neighbor} [{reason}]")
                    continue

                safe_speed = self.compute_safe_edge_speed(current.state, edge)
                step_cost = edge["distance_km"] if optimize_by == "distance" else (edge["distance_km"] / safe_speed)
                tentative_g = current.g_cost + step_cost

                if neighbor not in best_g or tentative_g < best_g[neighbor]:
                    best_g[neighbor] = tentative_g
                    h_val = self.kb.graph.get_heuristic(neighbor, goal, metric)
                    if optimize_by == "time":
                        h_val = h_val / 80.0
                    child = SearchTreeNode(
                        state=neighbor,
                        parent=current,
                        action=f"Drive {current.state} -> {neighbor} ({edge['distance_km']} km @ {safe_speed} km/h)",
                        g_cost=tentative_g,
                        h_cost=h_val
                    )
                    heapq.heappush(frontier, child)

        return {
            "success": False,
            "reason": "No valid safe route found satisfying trailer constraints.",
            "pruned_edges": pruned_edges
        }

    def evaluate_tactical_situation(self, percepts: Dict[str, Any]) -> Dict[str, Any]:
        """
        Step 6 & 7: Encodes current sensor observations and infers immediate driving actions.
        """
        self.kb.current_percepts = percepts
        light = percepts.get("traffic_light", "Green")
        for k in self.kb.signal_rules:
            if k.lower() == light.lower():
                light = k
                break

        location = percepts.get("current_node", "Unknown")
        obstacle_dist_m = percepts.get("obstacle_distance_m", 999.0)
        current_speed_kmh = percepts.get("current_speed_kmh", 60.0)
        steep_grade = percepts.get("steep_grade", False)
        weather = percepts.get("weather", self.kb.graph.nodes.get(location, {}).get("weather", "Clear"))

        mu = 0.7 if weather == "Clear" else 0.4
        v_ms = current_speed_kmh / 3.6
        stopping_distance_m = round((v_ms ** 2) / (2 * mu * 9.81) + (v_ms * 1.2), 1)

        decisions = []
        if obstacle_dist_m <= stopping_distance_m * 1.2:
            decisions.append(f"EMERGENCY_BRAKE: Obstacle at {obstacle_dist_m}m <= safe stopping threshold ({stopping_distance_m}m).")

        signal_action = self.kb.signal_rules.get(
            light, "UNKNOWN_SIGNAL — Engage hazard lights and execute controlled stop."
        )
        decisions.append(f"SIGNAL [{light}]: {signal_action}")

        if location in self.kb.graph.nodes and self.kb.graph.nodes[location]["weighbridge"]:
            if self.kb.trailer_profile["gross_weight_tons"] > 3.5:
                decisions.append(f"WEIGHBRIDGE [{location}]: Divert to inspection bay at 15 km/h.")

        mult = self.kb.weather_multipliers.get(weather, 0.7)
        target_speed = round(80.0 * mult, 1)
        if steep_grade:
            target_speed = min(target_speed, 45.0)
            decisions.append("GRADE_CONTROL: Steep descent detected — engage exhaust/engine brake, max 45 km/h.")
        decisions.append(f"TARGET_CRUISE_SPEED: {target_speed} km/h (Weather: {weather}, Stopping Dist: {stopping_distance_m}m)")

        return {
            "location": location,
            "percepts": percepts,
            "stopping_distance_m": stopping_distance_m,
            "actions": decisions
        }

agent = AutonomousTrailerAgent(kb)
print("AutonomousTrailerAgent initialized and ready for queries.")

AutonomousTrailerAgent initialized and ready for queries.


## Step 8: Querying the KB

In [6]:
# --- QUERY 1: Compare A* Search Tree across Distance Metrics ---
print("=== QUERY 1: A* ROUTE PLANNING (Mombasa_Port -> Eldoret_Terminal) ===")
for metric in ["haversine", "euclidean", "manhattan"]:
    result = agent.plan_route("Mombasa_Port", "Eldoret_Terminal", metric=metric, optimize_by="distance")
    print(f"\nHeuristic Metric: {metric.upper()}")
    print(f"  Optimal Path   : {' -> '.join(result['path'])}")
    print(f"  Total Distance : {result['total_cost']} km")
    print(f"  Nodes Expanded : {result['nodes_expanded']}")
    print(f"  Search Tree Trace (Node, g, h, f): {result['search_tree_trace']}")

# --- QUERY 2: Tactical Percept Inference ---
print("\n=== QUERY 2: TACTICAL DRIVING DECISION AT WEIGHBRIDGE ===")
percept_instance = {
    "current_node": "Voi_Weighbridge",
    "traffic_light": "Yellow",
    "weather": "Wet_Road",
    "current_speed_kmh": 65.0,
    "obstacle_distance_m": 120.0,
    "steep_grade": False
}
tactical_out = agent.evaluate_tactical_situation(percept_instance)
for step in tactical_out["actions"]:
    print("  ->", step)

# --- QUERY 3: Over-Height Bridge Constraint Pruning ---
print("\n=== QUERY 3: HEIGHT CLEARANCE CONSTRAINT REASONING ===")
kb.add_new_fact("trailer", "height_m", 4.55)
blocked_result = agent.plan_route("Machakos_Turnoff", "Nakuru_Hub", metric="haversine")
print("Result when trailer height is 4.55m:", json.dumps(blocked_result, indent=2))

kb.add_new_fact("trailer", "height_m", 4.30)

=== QUERY 1: A* ROUTE PLANNING (Mombasa_Port -> Eldoret_Terminal) ===

Heuristic Metric: HAVERSINE
  Optimal Path   : Mombasa_Port -> Voi_Weighbridge -> Mtito_Andei_Junction -> Machakos_Turnoff -> Nairobi_Southern_Bypass -> Naivasha_Dry_Port -> Nakuru_Hub -> Eldoret_Terminal
  Total Distance : 806.0 km
  Nodes Expanded : 9
  Search Tree Trace (Node, g, h, f): [('Mombasa_Port', 0.0, 704.06, 704.06), ('Voi_Weighbridge', 155.0, 567.86, 722.86), ('Mtito_Andei_Junction', 257.0, 480.23, 737.23), ('Machakos_Turnoff', 442.0, 322.02, 764.02), ('Nairobi_Southern_Bypass', 500.0, 262.08, 762.08), ('Naivasha_Dry_Port', 578.0, 188.21, 766.21), ('Nairobi_Inland_Depot', 494.0, 272.23, 766.23), ('Nakuru_Hub', 650.0, 127.97, 777.97), ('Eldoret_Terminal', 806.0, 0.0, 806.0)]

Heuristic Metric: EUCLIDEAN
  Optimal Path   : Mombasa_Port -> Voi_Weighbridge -> Mtito_Andei_Junction -> Machakos_Turnoff -> Nairobi_Southern_Bypass -> Naivasha_Dry_Port -> Nakuru_Hub -> Eldoret_Terminal
  Total Distance : 806.0 km

### Interactive Mode
Set `auto_demo=False` to type inputs directly.

In [7]:
def interactive_trailer_console(auto_demo: bool = True):
    """
    Interactive mode for the Autonomous Trailer Knowledge-Based Agent.
    Set auto_demo=False when running interactively in Jupyter/Colab to use input().
    """
    if auto_demo:
        print("[Auto-Demo Mode] Simulating dispatcher interactive session...")
        demo_inputs = [
            ("Mombasa_Port", "Kisumu_Port", "Flashing_Amber"),
            ("Nairobi_Southern_Bypass", "Nakuru_Hub", "Red")
        ]
        for orig, dest, sig in demo_inputs:
            print(f"\nDispatcher Query: Route {orig} -> {dest} | Observed Signal: {sig}")
            route = agent.plan_route(orig, dest, metric="haversine", optimize_by="time")
            if route["success"]:
                print(f"  Planned Path : {' -> '.join(route['path'])} (Est. Time: {route['total_cost']} hrs)")
            advice = agent.evaluate_tactical_situation({
                "current_node": orig,
                "traffic_light": sig,
                "current_speed_kmh": 55.0,
                "steep_grade": (orig == "Nairobi_Southern_Bypass")
            })
            print(f"  Immediate Action: {advice['actions'][0]}")
    else:
        print("Available Hubs:", list(kb.graph.nodes.keys()))
        orig = input("Enter origin hub: ").strip()
        dest = input("Enter destination hub: ").strip()
        sig = input("Enter current traffic light color: ").strip()
        route = agent.plan_route(orig, dest)
        print("Route Plan:", route)
        print("Tactical Advice:", agent.evaluate_tactical_situation({"current_node": orig, "traffic_light": sig}))

interactive_trailer_console(auto_demo=True)

[Auto-Demo Mode] Simulating dispatcher interactive session...

Dispatcher Query: Route Mombasa_Port -> Kisumu_Port | Observed Signal: Flashing_Amber
  Planned Path : Mombasa_Port -> Voi_Weighbridge -> Mtito_Andei_Junction -> Machakos_Turnoff -> Nairobi_Southern_Bypass -> Naivasha_Dry_Port -> Nakuru_Hub -> Kisumu_Port (Est. Time: 11.34 hrs)
  Immediate Action: SIGNAL [Flashing_Amber]: PROCEED_WITH_CAUTION — Reduce speed by 30% and monitor pedestrian/merging traffic.

Dispatcher Query: Route Nairobi_Southern_Bypass -> Nakuru_Hub | Observed Signal: Red
  Planned Path : Nairobi_Southern_Bypass -> Naivasha_Dry_Port -> Nakuru_Hub (Est. Time: 2.63 hrs)
  Immediate Action: SIGNAL [Red]: FULL_STOP — Engage air brakes and hold behind stop line.


## Step 9: Debugging the KB

In [8]:
print("--- DEBUGGING & STRESS-TESTING THE KNOWLEDGE BASE ---")

# 1. Faulty / Unknown Traffic Signal Test Cases (mirroring Exercise 3)
faulty_signals = ["Red", "Yellow", "Green", "Flashing_Red", "Blue", "Teal", "None"]
print("\n[Test 1] Traffic Signal Fault Tolerance:")
for sig in faulty_signals:
    res = agent.evaluate_tactical_situation({
        "current_node": "Mtito_Andei_Junction",
        "traffic_light": sig,
        "current_speed_kmh": 50.0,
        "obstacle_distance_m": 200.0
    })
    print(f"  Signal: {sig:14} -> {res['actions'][0]}")

# 2. Emergency Obstacle Within Braking Distance
print("\n[Test 2] Sudden Obstacle Braking Physics Check:")
emergency_res = agent.evaluate_tactical_situation({
    "current_node": "Nakuru_Hub",
    "traffic_light": "Green",
    "weather": "Heavy_Rain",
    "current_speed_kmh": 70.0,
    "obstacle_distance_m": 45.0
})
for act in emergency_res["actions"]:
    print(f"  {act}")

# 3. Dynamic Road Closure & Re-Routing Verification
print("\n[Test 3] Dynamic Highway Closure Handling:")
kb.set_road_status("Nakuru_Hub", "Kisumu_Port", open_road=False, reason="Bridge Maintenance")
reroute = agent.plan_route("Nakuru_Hub", "Kisumu_Port", metric="haversine")
print(f"  Direct link closed -> Agent automatically reroutes via: {' -> '.join(reroute['path'])} "
      f"({reroute['total_cost']} km)")
print(f"  Pruned edges logged: {reroute['pruned_edges']}")
kb.set_road_status("Nakuru_Hub", "Kisumu_Port", open_road=True)

--- DEBUGGING & STRESS-TESTING THE KNOWLEDGE BASE ---

[Test 1] Traffic Signal Fault Tolerance:
  Signal: Red            -> SIGNAL [Red]: FULL_STOP — Engage air brakes and hold behind stop line.
  Signal: Yellow         -> SIGNAL [Yellow]: DECELERATE_TO_STOP — Apply engine retarder; stop unless within minimum heavy-trailer braking distance.
  Signal: Green          -> SIGNAL [Green]: PROCEED — Scan cross-traffic, verify trailer swing clearance, and maintain target speed.
  Signal: Flashing_Red   -> SIGNAL [Flashing_Red]: STOP_AND_YIELD — Come to a complete stop, verify intersection clear, then proceed.
  Signal: Blue           -> SIGNAL [Blue]: UNKNOWN_SIGNAL — Engage hazard lights and execute controlled stop.
  Signal: Teal           -> SIGNAL [Teal]: UNKNOWN_SIGNAL — Engage hazard lights and execute controlled stop.
  Signal: None           -> SIGNAL [None]: UNCONTROLLED_INTERSECTION — Treat as yield; slow to 25 km/h and scan sensors.

[Test 2] Sudden Obstacle Braking Physics Check:


## Step 10: Reflection
- **What was tricky:** Mixing simple signal rules with heavy-truck stopping distances on wet roads or steep slopes, and keeping distance heuristics consistent across coordinate types. External APIs also occasionally time out, so fallbacks help keep runs predictable.
- **Improvements:** Add sensor uncertainty (like camera glare/fog), real-time traffic/signal timing, and multi-trailer coordination.

https://github.com/mbiriri/PS-Agents.git